# 🎭 FaceFusion Studio - Hoán Đổi Khuôn Mặt Video (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Runtime (Thời gian chạy)** -> **Change runtime type** -> Chọn **T4 GPU**.
> 3. Bấm **Runtime** -> **Run all (Chạy tất cả)**.
> 4. Khi Colab hỏi, chọn **'Kết nối với Google Drive'** (Connect to Google Drive). Lần đầu sẽ tải model và lưu vào Drive, **từ lần thứ 2 trở đi sẽ load tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title 1. Kết nối Google Drive & Kiểm tra Caching FaceFusion
import os
import shutil
import torch
from google.colab import drive
from IPython.display import clear_output

print("🔗 Đang kết nối với Google Drive của bạn...")
drive.mount('/content/drive')

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/FaceFusion"
output_drive_dir = "/content/drive/MyDrive/AI_Colab_Cache/FaceFusion_Outputs"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)
os.makedirs(output_drive_dir, exist_ok=True)

if torch.cuda.is_available():
    !sudo apt-get update -qq
    !sudo apt-get -y install cuda-toolkit -qq
    !apt-get install -qq libcudnn9-cuda-12
    device = "cuda"
    print("✅ Đang sử dụng Tesla T4 GPU")
else:
    device = "cpu"
    print("⚠️ Đang sử dụng CPU")

# Kiểm tra cache trong Google Drive
if os.path.exists(drive_cache_dir) and os.path.exists(f"{drive_cache_dir}/run.py"):
    print("🎉 ĐÃ TÌM THẤY FACEFUSION TRONG GOOGLE DRIVE! Bỏ qua tải về...")
    if not os.path.exists("/content/.program"):
        !cp -r "{drive_cache_dir}" /content/.program
else:
    print("⏳ Chưa có trong Drive. Đang tải FaceFusion lần đầu và lưu vào Google Drive của bạn...")
    !git clone https://github.com/facefusion/facefusion /content/.program --single-branch
    %cd /content/.program
    if os.path.exists("facefusion.py"):
        os.rename("facefusion.py", "run.py")
    print("💾 Đang lưu bản sao vào Google Drive để lần sau không phải tải lại...")
    !cp -r /content/.program "{drive_cache_dir}"

%cd /content/.program
if device == "cuda":
    !python install.py --onnxruntime cuda --skip-conda
else:
    !python install.py --onnxruntime default --skip-conda

clear_output()
print("🎉 Cài đặt hoàn tất! Đang khởi động WebUI FaceFusion...")


In [ ]:
#@title 2. Khởi chạy Giao diện Web (WebUI Gradio & Cloudflare)
import os
import time
from IPython.display import clear_output

%cd /content/.program
output_drive_dir = "/content/drive/MyDrive/AI_Colab_Cache/FaceFusion_Outputs"

# Bật chế độ tạo link công khai Gradio
!sed -i 's/launch(f/launch(share=True, f/g' /content/.program/facefusion/uis/layouts/default.py 2>/dev/null || true

# Tải Cloudflare Tunnel dự phòng
!curl -LOs https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
!nohup cloudflared tunnel --url localhost:7860 > nohup.out 2>&1 &

time.sleep(3)
cf_url = !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" nohup.out
if cf_url:
    print("🔗 Đường link Cloudflare Tunnel (Dự phòng):", cf_url[0])

print("🚀 Đang khởi động FaceFusion WebUI (kết quả sẽ tự động lưu vào Google Drive: AI_Colab_Cache/FaceFusion_Outputs)...")
!python run.py run --execution-providers cuda --output-path "{output_drive_dir}" --open-browser
